# Procurement Agent — Unit Tests

Tests `run()` from `procurement.ipynb` using the **HuggingFace Inference API**.

**Run order:**
1. Setup (sys.path + load .env + run procurement notebook to register `run()`)
2. Mock stylist outputs (three aesthetics)
3. Helper definitions
4. First-pass test — all three styles, no critic feedback
5. Retry-pass tests — cottagecore and japandi with critic feedback

## Setup

In [ ]:
import sys
import json
import os
from pathlib import Path

# Resolve src/ so agents.* are importable
src_dir = str(Path(".").resolve().parent / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Load .env from repo root
try:
    from dotenv import load_dotenv
    repo_root = Path(".").resolve().parent
    load_dotenv(repo_root / ".env")
except Exception:
    pass

# Execute procurement.ipynb in this kernel — sets up InferenceClient and makes run() available
%run ../src/agents/procurement.ipynb

## Mock stylist outputs — three distinct aesthetics

In [4]:
STYLE_COTTAGECORE = {
    "style_profile": (
        "Bright spring garden vibe: airy pastels, natural textures, "
        "cottagecore-meets-modern sensibility, light wood accents, and a "
        "relaxed outdoor-living feeling."
    ),
    "aesthetic": ["cottagecore", "boho outdoor", "spring patio", "whimsical garden"],
    "colors": ["sage green", "cream", "terracotta"],
    "materials": ["ceramic", "rattan", "wood"],
    "budget_max": 75.0,
    "budget_currency": "USD",
}

STYLE_JAPANDI = {
    "style_profile": (
        "Calm, stripped-back Japandi interior: warm neutrals, clean lines, "
        "wabi-sabi imperfection, natural materials like linen and pale oak, "
        "and a philosophy of intentional minimalism over decoration."
    ),
    "aesthetic": ["japandi", "wabi-sabi", "zen minimalist", "nordic calm"],
    "colors": ["warm white", "pale oak", "charcoal", "stone grey"],
    "materials": ["linen", "oak", "bamboo", "matte ceramic"],
    "budget_max": 150.0,
    "budget_currency": "USD",
}

STYLE_Y2K = {
    "style_profile": (
        "Nostalgic Y2K / early 2000s revival: iridescent and holographic "
        "surfaces, bubblegum pinks, chrome accents, low-rise silhouettes, "
        "butterfly motifs, and playful maximalism with a futuristic edge."
    ),
    "aesthetic": ["Y2K", "cyber Y2K", "2000s nostalgia", "pop maximalist"],
    "colors": ["bubblegum pink", "chrome silver", "electric blue", "white"],
    "materials": ["holographic vinyl", "patent leather", "mesh", "acrylic"],
    "budget_max": 60.0,
    "budget_currency": "USD",
}

STYLES = {
    "cottagecore": STYLE_COTTAGECORE,
    "japandi":     STYLE_JAPANDI,
    "y2k":         STYLE_Y2K,
}

REQUIRED_PRODUCT_KEYS = {"image_url", "product_name", "price", "link", "tags"}

## Helpers

In [5]:
def _check_env():
    missing = [k for k in ("SERPAPI_API_KEY", "HF_TOKEN") if not os.environ.get(k)]
    if missing:
        raise RuntimeError(
            f"Missing required environment variables: {', '.join(missing)}. "
            "Set them in .env or the environment."
        )


def _assert_product_shape(items: list, label: str = ""):
    assert isinstance(items, list), f"[{label}] procurement_products must be a list"
    assert len(items) > 0, f"[{label}] procurement_products must not be empty"
    for it in items:
        assert set(it.keys()) == REQUIRED_PRODUCT_KEYS, (
            f"[{label}] Unexpected product keys: {set(it.keys())}"
        )
        assert isinstance(it["image_url"], str)
        assert isinstance(it["product_name"], str)
        assert it["price"] is None or isinstance(it["price"], float)
        assert isinstance(it["link"], str)
        assert " " not in it["link"], f"[{label}] URL contains spaces: {it['link']}"
        assert isinstance(it["tags"], list)


def _run_style(stylist_output: dict, critic_feedback: str = None) -> dict:
    state = {
        "stylist_output": stylist_output,
        "num_queries": 3,
        "results_per_query": 5,  # keep low for test speed
    }
    if critic_feedback:
        state["critic_feedback"] = critic_feedback
    return json.loads(run(state))


print("Helpers defined.")

Helpers defined.


## Test 1 — All styles, first pass (no critic feedback)

Runs all three styles through the procurement agent with no critic feedback.
Verifies each returns LLM-generated queries and a valid candidate pool.

In [6]:
# test_all_styles_first_pass
_check_env()

for name, style in STYLES.items():
    print(f"\n{'='*60}")
    print(f"Style: {name.upper()}")
    print(f"{'='*60}")

    result = _run_style(style)

    queries = result.get("procurement_queries", [])
    print("Queries:\n" + json.dumps(queries, indent=2))
    assert isinstance(queries, list), f"[{name}] queries must be a list"
    assert len(queries) == 3, f"[{name}] expected 3 queries, got {len(queries)}"
    assert all(isinstance(q, str) and q.strip() for q in queries), (
        f"[{name}] all queries must be non-empty strings"
    )

    assert result.get("style_profile") == style["style_profile"], (
        f"[{name}] style_profile must pass through unchanged"
    )

    items = result.get("procurement_products", [])
    print(f"Total candidates: {len(items)}")
    stride = 5  # matches results_per_query in _run_style
    for i, q in enumerate(queries):
        sample = items[i * stride] if i * stride < len(items) else None
        print(f"\n  [{i+1}] Query: {q}")
        print(f"       Sample: {json.dumps(sample, indent=2)}")
    _assert_product_shape(items, label=name)

print("\nPASS  test_all_styles_first_pass")


Style: COTTAGECORE
[procurement] Calling HF Inference API (model: Qwen/Qwen2.5-7B-Instruct)...
[procurement] Query generation complete. Raw response: ["sage green ceramic vase spring garden vibe", "wooden outdoor side table cottagecore meets modern", "terracotta flower pot whimsical garden decor"]
[procurement] Fetching products for 3 queries (up to 5 results each)...
[procurement]   [1/3] Querying SerpAPI: 'sage green ceramic vase spring garden vibe'
[procurement]          → 5 new products (pool total: 5)
[procurement]   [2/3] Querying SerpAPI: 'wooden outdoor side table cottagecore meets modern'
[procurement]          → 5 new products (pool total: 10)
[procurement]   [3/3] Querying SerpAPI: 'terracotta flower pot whimsical garden decor'
[procurement]          → 5 new products (pool total: 15)
[procurement] Done. Total candidate pool: 15 products.
Queries:
[
  "sage green ceramic vase spring garden vibe",
  "wooden outdoor side table cottagecore meets modern",
  "terracotta flower po

## Test 2 — Cottagecore with critic feedback

Retry-pass test using the cottagecore style.
Verifies the agent returns a valid pool when `critic_feedback` is present.

In [7]:
# test_cottagecore_with_critic_feedback
_check_env()

result = _run_style(
    STYLE_COTTAGECORE,
    critic_feedback=(
        "Outdoor lighting results scored 0.28 avg — too industrial/modern. "
        "Garden decor results scored 0.31 avg — too generic. "
        "Planters scored 0.79 avg — keep the same approach."
    ),
)

queries = result.get("procurement_queries", [])
print("\nQueries (with critic feedback):\n" + json.dumps(queries, indent=2))
assert len(queries) == 3
assert all(isinstance(q, str) and q.strip() for q in queries)

items = result.get("procurement_products", [])
print(f"Total candidates: {len(items)}")
_assert_product_shape(items, label="cottagecore_retry")

print("\nPASS  test_cottagecore_with_critic_feedback")

[procurement] Calling HF Inference API (model: Qwen/Qwen2.5-7B-Instruct)...
[procurement] Retry pass — critic feedback:
  Outdoor lighting results scored 0.28 avg — too industrial/modern. Garden decor results scored 0.31 avg — too generic. Planters scored 0.79 avg — keep the same approach.
[procurement] Query generation complete. Raw response: ["sage green ceramic vase spring patio", "wooden rattan outdoor chair cottagecore", "terracotta flower pot whimsical garden"]
[procurement] Fetching products for 3 queries (up to 5 results each)...
[procurement]   [1/3] Querying SerpAPI: 'sage green ceramic vase spring patio'
[procurement]          → 5 new products (pool total: 5)
[procurement]   [2/3] Querying SerpAPI: 'wooden rattan outdoor chair cottagecore'
[procurement]          → 5 new products (pool total: 10)
[procurement]   [3/3] Querying SerpAPI: 'terracotta flower pot whimsical garden'
[procurement]          → 5 new products (pool total: 15)
[procurement] Done. Total candidate pool: 15

## Test 3 — Japandi with critic feedback

Retry-pass test using the japandi style.

In [8]:
# test_japandi_with_critic_feedback
_check_env()

result = _run_style(
    STYLE_JAPANDI,
    critic_feedback=(
        "Furniture results scored 0.25 avg — too ornate, not minimal enough. "
        "Textiles scored 0.71 avg — good, keep this angle. "
        "Ceramics scored 0.38 avg — too decorative, need plainer forms."
    ),
)

queries = result.get("procurement_queries", [])
print("\nQueries (with critic feedback):\n" + json.dumps(queries, indent=2))
assert len(queries) == 3
assert all(isinstance(q, str) and q.strip() for q in queries)

items = result.get("procurement_products", [])
print(f"Total candidates: {len(items)}")
_assert_product_shape(items, label="japandi_retry")

print("\nPASS  test_japandi_with_critic_feedback")

[procurement] Calling HF Inference API (model: Qwen/Qwen2.5-7B-Instruct)...
[procurement] Retry pass — critic feedback:
  Furniture results scored 0.25 avg — too ornate, not minimal enough. Textiles scored 0.71 avg — good, keep this angle. Ceramics scored 0.38 avg — too decorative, need plainer forms.
[procurement] Query generation complete. Raw response: ["linen throw pillow wabi-sabi imperfection", "oak coffee table nordic calm", "bamboo wall art minimal zen"]
[procurement] Fetching products for 3 queries (up to 5 results each)...
[procurement]   [1/3] Querying SerpAPI: 'linen throw pillow wabi-sabi imperfection'
[procurement]          → 5 new products (pool total: 5)
[procurement]   [2/3] Querying SerpAPI: 'oak coffee table nordic calm'
[procurement]          → 5 new products (pool total: 10)
[procurement]   [3/3] Querying SerpAPI: 'bamboo wall art minimal zen'
[procurement]          → 5 new products (pool total: 15)
[procurement] Done. Total candidate pool: 15 products.

Queries (w